# Bases de données — Exercice 2 : les aéroports (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (requête SQL, code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** `SELECT ... FROM ... WHERE`, `DISTINCT`, `COUNT`, `MAX`, `MIN`, `AND`, `LIKE`, jointures (`JOIN ... ON`), requêtes imbriquées, `ORDER BY`, et pour finir `GROUP BY`.

**Données :** le dossier `airports_sql/` (contenu de l'archive `airports_sql.zip` du site du professeur) doit être **dans le même dossier** que ce notebook.

**Question 5 :** le fichier `airports_tables_v2.sql` n'est pas encore disponible sur le site du professeur (lien cassé). La question 5 n'est donc pas traitée ; les questions de la partie 6 n'utilisent que des attributs déjà présents dans les tables, elles sont résolues avec les données actuelles.

Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

---
# MISE EN PLACE

En classe, les requêtes sont tapées dans la **console H2** (un SGBD). Ici, on utilise le module `sqlite3` de Python, qui contient un autre SGBD (**SQLite**) : **le langage SQL est le même**, donc chaque requête de ce notebook peut être recopiée telle quelle dans la console H2 (les rares différences sont signalées).

- `execute_fichier("fichier.sql")` joue le rôle de `RUNSCRIPT FROM 'fichier.sql';` dans H2 ;
- `affiche(requete)` exécute une requête et affiche le résultat sous forme de tableau, comme la console H2 ;
- `resultat(requete)` renvoie la liste des enregistrements : elle ne sert qu'aux vérifications.

La requête SQL est écrite entre trois guillemets `""" ... """` : c'est une chaîne de caractères Python qui peut tenir sur plusieurs lignes.

In [1]:
import sqlite3

base = sqlite3.connect(":memory:")   # une base de données vide, en mémoire vive


def execute_fichier(nom_fichier):
    """Équivalent de RUNSCRIPT FROM '...' dans H2 : exécute un fichier .sql."""
    fichier = open(nom_fichier, encoding="utf-8")
    script = fichier.read()
    fichier.close()
    base.executescript(script)


def resultat(requete):
    """Renvoie la liste des enregistrements obtenus (pour les vérifications)."""
    return base.execute(requete).fetchall()


def affiche(requete, maximum=20):
    """Exécute une requête et affiche le résultat comme dans la console H2."""
    curseur = base.execute(requete)
    if curseur.description is None:          # INSERT, UPDATE, DELETE, CREATE...
        print("Instruction exécutée.")
        return
    entete = ""
    for colonne in curseur.description:
        entete = entete + colonne[0] + " | "
    print(entete)
    nombre = 0
    for ligne in curseur:
        if nombre < maximum:
            texte = ""
            for valeur in ligne:
                texte = texte + str(valeur) + " | "
            print(texte)
        nombre = nombre + 1
    if nombre > maximum:
        print("... (seules les " + str(maximum) + " premières lignes sont affichées)")
    print("(" + str(nombre) + " ligne(s))")

---
# QUESTION 1

#### 📝 ÉNONCÉ

> 1. Télécharger l'archive airports_sql.zip : il contient les fichiers airports_tables.sql et airports_data.sql : exécuter ces fichiers dans cet ordre dans la console H2.

#### ✅ RÉSOLUTION

Dans la console H2 (en remplaçant le chemin par celui du dossier où l'archive a été décompressée) :

```sql
RUNSCRIPT FROM '/chemin/vers/airports_sql/airports_tables.sql';
RUNSCRIPT FROM '/chemin/vers/airports_sql/airports_data.sql';
```

⚠️ **Important dans H2 (non demandé, mais indispensable) :** exécuter aussi ces quatre lignes, une seule fois après le chargement. Sans elles, les requêtes avec jointure (questions 3 (j), 4 et 6) peuvent prendre **plusieurs minutes** dans H2 (voir l'explication ci-dessous) :

```sql
CREATE INDEX ON Countries(code);
CREATE INDEX ON Airports(id);
CREATE INDEX ON Regions(code);
CREATE INDEX ON Runways(id);
```

Dans le notebook :

In [2]:
execute_fichier("airports_sql/airports_tables.sql")
execute_fichier("airports_sql/airports_data.sql")

affiche("SELECT COUNT(*) FROM Countries")
affiche("SELECT COUNT(*) FROM Regions")
affiche("SELECT COUNT(*) FROM Airports")
affiche("SELECT COUNT(*) FROM Runways")

COUNT(*) | 
248 | 
(1 ligne(s))
COUNT(*) | 
3862 | 
(1 ligne(s))
COUNT(*) | 
73904 | 
(1 ligne(s))
COUNT(*) | 
44006 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [3]:
assert resultat("SELECT COUNT(*) FROM Countries") == [(248,)]
assert resultat("SELECT COUNT(*) FROM Airports") == [(73904,)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Pourquoi cet ordre ?** `airports_tables.sql` contient les instructions `CREATE TABLE` : il décrit la **structure** de la base (le *schéma* : les relations et leurs attributs). `airports_data.sql` contient des milliers d'instructions `INSERT INTO` : c'est le **contenu**. On ne peut pas ranger des données dans une table qui n'existe pas encore : il faut d'abord créer les tables.
- C'est exactement la distinction du cours : **structure** d'une base de données (le schéma) ≠ **contenu** (les enregistrements).
- Le fichier `airports_tables.sql` commence par des `DROP TABLE IF EXISTS ...;` : si les tables existent déjà (par exemple après l'exercice 1), elles sont supprimées puis recréées. On peut donc relancer les deux fichiers sans créer de doublons.
- On obtient 248 pays, 3 862 régions, 73 904 aéroports et 44 006 pistes.
- **Pourquoi les index ?** Les tables de `airports_tables.sql` n'ont **pas de clé primaire**. Pour une jointure entre `Runways` (44 006 pistes) et `Airports` (73 904 aéroports), H2 compare alors chaque piste à chaque aéroport : plus de 3 milliards de comparaisons ! Un **index** est une sorte de table des matières : il permet au SGBD de trouver directement l'aéroport d'identifiant donné, sans tout parcourir. Avec les index, la même requête passe de plusieurs minutes à une fraction de seconde. C'est un des services rendus par un SGBD cités dans le cours : l'**efficacité de traitement des requêtes**. (SQLite, utilisé dans ce notebook, crée ces index tout seul pendant les jointures ; H2 non.)
- Déclarer une **clé primaire** crée automatiquement un index : c'est sans doute l'un des buts du fichier `airports_tables_v2.sql` de la question 5. `CREATE INDEX` est hors programme.

---
# QUESTION 2

> La commande SELECT permet d'extraire des données des relations (tables) de la base de données.
> Donner la syntaxe des instructions qui permettent d'afficher les données ci-dessous.

## 2 (a)

#### 📝 ÉNONCÉ

> (a) La relation Countries.

#### ✅ RÉSOLUTION

```sql
SELECT * FROM Countries;
```

*Exécution dans le notebook :*

In [4]:
affiche("""
SELECT * FROM Countries;
""")

id | code | name | continent | 
302672 | AD | Andorra | EU | 
302618 | AE | United Arab Emirates | AS | 
302619 | AF | Afghanistan | AS | 
302722 | AG | Antigua and Barbuda | NA | 
302723 | AI | Anguilla | NA | 
302673 | AL | Albania | EU | 
302620 | AM | Armenia | AS | 
302556 | AO | Angola | AF | 
302615 | AQ | Antarctica | AN | 
302789 | AR | Argentina | SA | 
302763 | AS | American Samoa | OC | 
302674 | AT | Austria | EU | 
302764 | AU | Australia | OC | 
302725 | AW | Aruba | NA | 
302621 | AZ | Azerbaijan | AS | 
302675 | BA | Bosnia and Herzegovina | EU | 
302726 | BB | Barbados | NA | 
302622 | BD | Bangladesh | AS | 
302676 | BE | Belgium | EU | 
302557 | BF | Burkina Faso | AF | 
... (seules les 20 premières lignes sont affichées)
(248 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- L'étoile `*` signifie « **tous les attributs** ». Sans `WHERE`, on obtient **tous les enregistrements** : les 248 pays (le notebook n'en affiche que 20).
- Chaque ligne est un **enregistrement** (un pays), chaque colonne un **attribut** (`id`, `code`, `name`, `continent`).

## 2 (b)

#### 📝 ÉNONCÉ

> (b) Les noms de tous les pays enregistrés dans la relation Countries.

#### ✅ RÉSOLUTION

```sql
SELECT name FROM Countries;
```

*Exécution dans le notebook :*

In [5]:
affiche("""
SELECT name FROM Countries;
""")

name | 
Andorra | 
United Arab Emirates | 
Afghanistan | 
Antigua and Barbuda | 
Anguilla | 
Albania | 
Armenia | 
Angola | 
Antarctica | 
Argentina | 
American Samoa | 
Austria | 
Australia | 
Aruba | 
Azerbaijan | 
Bosnia and Herzegovina | 
Barbados | 
Bangladesh | 
Belgium | 
Burkina Faso | 
... (seules les 20 premières lignes sont affichées)
(248 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- Après `SELECT`, on écrit **les attributs que l'on veut voir** : ici seulement `name`. Le résultat a toujours 248 lignes, mais une seule colonne.
- ⚠️ Les noms des attributs sont ceux du schéma, **en anglais** (`name`, pas `nom`).

## 2 (c)

#### 📝 ÉNONCÉ

> (c) Les noms et les codes ISO de tous les pays renseignés dans la relation Countries.

#### ✅ RÉSOLUTION

```sql
SELECT name, code FROM Countries;
```

*Exécution dans le notebook :*

In [6]:
affiche("""
SELECT name, code FROM Countries;
""")

name | code | 
Andorra | AD | 
United Arab Emirates | AE | 
Afghanistan | AF | 
Antigua and Barbuda | AG | 
Anguilla | AI | 
Albania | AL | 
Armenia | AM | 
Angola | AO | 
Antarctica | AQ | 
Argentina | AR | 
American Samoa | AS | 
Austria | AT | 
Australia | AU | 
Aruba | AW | 
Azerbaijan | AZ | 
Bosnia and Herzegovina | BA | 
Barbados | BB | 
Bangladesh | BD | 
Belgium | BE | 
Burkina Faso | BF | 
... (seules les 20 premières lignes sont affichées)
(248 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- Plusieurs attributs sont **séparés par des virgules**. Les colonnes du résultat apparaissent **dans l'ordre où on les écrit** (ici le nom, puis le code), pas dans l'ordre de la table.

## 2 (d)

#### 📝 ÉNONCÉ

> (d) Les codes des continents renseignés dans la relation Countries.

#### ✅ RÉSOLUTION

```sql
SELECT DISTINCT continent FROM Countries;
```

**Réponse :** 7 codes : `EU`, `AS`, `NA`, `AF`, `AN`, `SA`, `OC`.

*Exécution dans le notebook :*

In [7]:
affiche("""
SELECT DISTINCT continent FROM Countries;
""")

continent | 
EU | 
AS | 
NA | 
AF | 
AN | 
SA | 
OC | 
(7 ligne(s))


*Vérification (non demandée) :*

In [8]:
print(len(resultat("SELECT continent FROM Countries")), "lignes sans DISTINCT")
print(len(resultat("SELECT DISTINCT continent FROM Countries")), "lignes avec DISTINCT")
assert len(resultat("SELECT DISTINCT continent FROM Countries")) == 7

248 lignes sans DISTINCT
7 lignes avec DISTINCT


#### 💡 EXPLICATION PÉDAGOGIQUE

- Sans `DISTINCT`, on obtiendrait **248 lignes** (une par pays) : `EU` apparaîtrait des dizaines de fois.
- Le mot-clé `DISTINCT` **supprime les doublons** du résultat : chaque code n'apparaît qu'une fois.
- Il se place **juste après `SELECT`**.

## 2 (e)

#### 📝 ÉNONCÉ

> (e) L'identifiant de la France dans la relation Countries.

#### ✅ RÉSOLUTION

```sql
SELECT id FROM Countries WHERE name = 'France';
```

**Réponse :** 302687.

*Exécution dans le notebook :*

In [9]:
affiche("""
SELECT id FROM Countries WHERE name = 'France';
""")

id | 
302687 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [10]:
assert resultat("SELECT id FROM Countries WHERE name = 'France'") == [(302687,)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- La clause `WHERE` **filtre** les enregistrements : on ne garde que ceux qui vérifient la condition.
- En SQL, le test d'égalité s'écrit avec **un seul** `=` (et non `==` comme en Python).
- Les chaînes de caractères sont entre **guillemets simples** `'France'`. ⚠️ Pas d'espace à l'intérieur : `' France '` serait une autre chaîne (avec des espaces) et la requête ne trouverait rien. C'est le piège du copier-coller depuis le PDF.
- On aurait aussi pu écrire `WHERE code = 'FR'`.

## 2 (f)

#### 📝 ÉNONCÉ

> (f) Les coordonnées GPS de l'aéroport d'Orly.

#### ✅ RÉSOLUTION

On commence par **chercher** l'aéroport, car on ne connaît pas son nom exact ni son code :

```sql
SELECT code, name, city FROM Airports WHERE name LIKE '%Orly%';
```

Trois aéroports contiennent « Orly » : celui de Paris a le code `LFPO`. D'où la requête :

```sql
SELECT latitude, longitude FROM Airports WHERE code = 'LFPO';
```

**Réponse :** latitude 48.7233333, longitude 2.3794444.

*Exécution dans le notebook :*

In [11]:
affiche("""
SELECT latitude, longitude FROM Airports WHERE code = 'LFPO';
""")

latitude | longitude | 
48.7233333 | 2.3794444 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [12]:
affiche("SELECT code, name, city FROM Airports WHERE name LIKE '%Orly%'")
assert resultat("SELECT latitude, longitude FROM Airports WHERE code = 'LFPO'") == [(48.7233333, 2.3794444)]

code | name | city | 
KE-0034 | Orly (Olooitikosh) Airport | Orly (Olooitikosh) | 
LFPO | Paris-Orly Airport | Paris | 
RU-0163 | Yegorlykskaya Air Base | Yegorlykskaya | 
(3 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- Les coordonnées GPS sont deux attributs de la relation `Airports` : `latitude` et `longitude`.
- **`LIKE` avec `%`** permet de chercher un *morceau* de chaîne : `'%Orly%'` signifie « n'importe quoi, puis Orly, puis n'importe quoi ». C'est très pratique pour explorer les données.
- ⚠️ **Piège :** `WHERE name LIKE '%Orly%'` renvoie **3 aéroports**, dont un au Kenya (*Orly (Olooitikosh) Airport*) et une base fermée en Russie. Il faut ensuite filtrer sur une valeur **unique** : le code ICAO `'LFPO'` (ou le nom exact `'Paris-Orly Airport'`).
- ⚠️ Dans H2, `LIKE` **distingue majuscules et minuscules** : `'%orly%'` ne trouverait rien. (SQLite, lui, ne les distingue pas.)

## 2 (g)

#### 📝 ÉNONCÉ

> (g) Le nombre de régions renseignées dans la relation Regions.

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Regions;
```

**Réponse :** 3 862 régions.

*Exécution dans le notebook :*

In [13]:
affiche("""
SELECT COUNT(*) FROM Regions;
""")

COUNT(*) | 
3862 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [14]:
assert resultat("SELECT COUNT(*) FROM Regions") == [(3862,)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- `COUNT(*)` est une **fonction d'agrégation** : au lieu d'afficher les enregistrements, elle les **compte**. Le résultat est une seule valeur.
- `COUNT(*)` compte les lignes ; `COUNT(name)` compterait les lignes où `name` est renseigné (non `NULL`). Ici, c'est pareil.

## 2 (h)

#### 📝 ÉNONCÉ

> (h) La plus grande longueur de piste renseignée dans la relation Runways.

#### ✅ RÉSOLUTION

```sql
SELECT MAX(length) FROM Runways;
```

**Réponse :** 30 000.

*Exécution dans le notebook :*

In [15]:
affiche("""
SELECT MAX(length) FROM Runways;
""")

MAX(length) | 
30000 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [16]:
assert resultat("SELECT MAX(length) FROM Runways") == [(30000,)]
affiche("SELECT * FROM Runways WHERE length = 30000")

id | airport_id | type | length | width | lighted | closed | 
256946 | 22159 | WATER | 30000 | 5000 | 0 | 0 | 
(1 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- `MAX` est une autre fonction d'agrégation (comme `MIN`, `SUM`, `AVG`, `COUNT`).
- **Unité :** le jeu de données *OurAirports* donne les longueurs et largeurs **en pieds** (1 pied ≈ 0,3048 m). La plus longue piste de Charles de Gaulle mesure 13 829 pieds, soit 4 215 m.
- **Regard critique :** la piste de 30 000 pieds (plus de 9 km !) est une piste **sur l'eau** (`type = 'WATER'`) d'une base d'hydravions aux États-Unis : ce n'est pas une vraie piste en dur. Une donnée peut être juste pour le SGBD et pourtant surprenante : il faut toujours **regarder** ses résultats.

---
# QUESTION 3

> En utilisant le jeu de données fourni, répondre aux questions suivantes à l'aide de requêtes SQL.

Dans cette question, on s'autorise **plusieurs requêtes successives** quand c'est naturel (on cherche d'abord un code ou un identifiant, puis on l'utilise). La question 4 demande ensuite de tout faire en **une seule** requête.

## 3 (a)

#### 📝 ÉNONCÉ

> (a) Quel est le pays dont le code ISO est IM ?

#### ✅ RÉSOLUTION

```sql
SELECT name FROM Countries WHERE code = 'IM';
```

**Réponse :** *Isle of Man* (l'île de Man).

*Exécution dans le notebook :*

In [17]:
affiche("""
SELECT name FROM Countries WHERE code = 'IM';
""")

name | 
Isle of Man | 
(1 ligne(s))


*Vérification (non demandée) :*

In [18]:
assert resultat("SELECT name FROM Countries WHERE code = 'IM'") == [("Isle of Man",)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- Requête de base : on connaît la valeur d'un attribut (`code`), on veut la valeur d'un autre attribut (`name`) **du même enregistrement**.

## 3 (b)

#### 📝 ÉNONCÉ

> (b) Quel est le code ICAO de l'aéroport de Toulouse-Blagnac ?

#### ✅ RÉSOLUTION

On cherche d'abord le nom exact (les noms sont en anglais) :

```sql
SELECT code, name FROM Airports WHERE name LIKE '%Blagnac%';
```

puis :

```sql
SELECT code FROM Airports WHERE name = 'Toulouse-Blagnac Airport';
```

**Réponse :** `LFBO`.

*Exécution dans le notebook :*

In [19]:
affiche("""
SELECT code FROM Airports WHERE name = 'Toulouse-Blagnac Airport';
""")

code | 
LFBO | 
(1 ligne(s))


*Vérification (non demandée) :*

In [20]:
affiche("SELECT code, name FROM Airports WHERE name LIKE '%Blagnac%'")
assert resultat("SELECT code FROM Airports WHERE name = 'Toulouse-Blagnac Airport'") == [("LFBO",)]

code | name | 
LFBO | Toulouse-Blagnac Airport | 
(1 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- Dans la relation `Airports`, l'attribut `code` est le code **ICAO** de l'aéroport (4 lettres ; ceux de France commencent par `LF`).
- `WHERE name LIKE '%Toulouse%'` renverrait **5 aéroports** (Toulouse-Francazal, Toulouse-Lasbordes...). Chercher `'%Blagnac%'` est plus précis.
- La première requête suffit d'ailleurs à répondre ; la seconde montre la requête « propre » avec le nom exact.

## 3 (c)

#### 📝 ÉNONCÉ

> (c) Quels sont les noms des héliports de la capitale de l'Allemagne ?

#### ✅ RÉSOLUTION

Le code ISO de l'Allemagne s'obtient avec `SELECT code FROM Countries WHERE name = 'Germany';` → `DE`. Puis :

```sql
SELECT name FROM Airports WHERE type = 'heliport' AND city = 'Berlin' AND country = 'DE';
```

**Réponse :** 8 héliports (Berlin Emergency Hospital Heliport, Charité Clinic Virchow Campus Heliport, Berlin Tempelhof Heliport, ...).

*Exécution dans le notebook :*

In [21]:
affiche("""
SELECT name FROM Airports WHERE type = 'heliport' AND city = 'Berlin' AND country = 'DE';
""")

name | 
Berlin Emergency Hospital Heliport | 
Helios Clinic Berlin-Buch Heliport | 
Vivantes Humboldt Clinic Heliport | 
Charité Clinic Virchow Campus Heliport | 
Vivantes Freidrichshain Clinic Heliport | 
Berlin Tempelhof Heliport | 
Charité Clinic Benjamin Franklin Campus Heliport | 
Havelhöhe Medical Heliport | 
(8 ligne(s))


*Vérification (non demandée) :*

In [22]:
assert resultat("SELECT code FROM Countries WHERE name = 'Germany'") == [("DE",)]
assert len(resultat("SELECT name FROM Airports WHERE type = 'heliport' AND city = 'Berlin' AND country = 'DE'")) == 8

# Sans la condition sur le pays :
affiche("SELECT name, country FROM Airports WHERE type = 'heliport' AND city = 'Berlin'")

name | country | 
Berlin Fairgrounds Heliport | US | 
AGH Heliport | US | 
Androscoggin Valley Hospital Helipad | US | 
Berlin Emergency Hospital Heliport | DE | 
Helios Clinic Berlin-Buch Heliport | DE | 
Vivantes Humboldt Clinic Heliport | DE | 
Charité Clinic Virchow Campus Heliport | DE | 
Vivantes Freidrichshain Clinic Heliport | DE | 
Berlin Tempelhof Heliport | DE | 
Charité Clinic Benjamin Franklin Campus Heliport | DE | 
Havelhöhe Medical Heliport | DE | 
West Jersey Hospital Heliport | US | 
(12 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- On combine plusieurs conditions avec **`AND`** : un enregistrement est gardé seulement si **toutes** les conditions sont vraies.
- Les types d'aéroports sont écrits en anglais et en minuscules : `'heliport'`, `'small_airport'`, etc. (voir `SELECT DISTINCT type FROM Airports;`).
- ⚠️ **Le piège :** sans `country = 'DE'`, on obtient **12** héliports, car il existe aussi des villes nommées *Berlin* aux **États-Unis** (New Hampshire, New Jersey...). Une ville n'est pas identifiée par son seul nom : il faut préciser le pays.

## 3 (d)

#### 📝 ÉNONCÉ

> (d) Combien de régions sont répertoriées au Brésil ?

#### ✅ RÉSOLUTION

Code du Brésil : `SELECT code FROM Countries WHERE name = 'Brazil';` → `BR`. Puis :

```sql
SELECT COUNT(*) FROM Regions WHERE country = 'BR';
```

**Réponse :** 28 régions (dont une région « (unassigned) », donc 27 vraies régions : les 26 États et le District fédéral).

*Exécution dans le notebook :*

In [23]:
affiche("""
SELECT COUNT(*) FROM Regions WHERE country = 'BR';
""")

COUNT(*) | 
28 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [24]:
assert resultat("SELECT COUNT(*) FROM Regions WHERE country = 'BR'") == [(28,)]
affiche("SELECT code, name FROM Regions WHERE country = 'BR'", 30)

code | name | 
BR-AC | Acre | 
BR-AL | Alagoas | 
BR-AM | Amazonas | 
BR-AP | Amapá | 
BR-BA | Bahia | 
BR-CE | Ceará | 
BR-DF | Distrito Federal | 
BR-ES | Espírito Santo | 
BR-GO | Goiás | 
BR-MA | Maranhão | 
BR-MG | Minas Gerais | 
BR-MS | Mato Grosso do Sul | 
BR-MT | Mato Grosso | 
BR-PA | Pará | 
BR-PB | Paraíba | 
BR-PE | Pernambuco | 
BR-PI | Piauí | 
BR-PR | Paraná | 
BR-RJ | Rio de Janeiro | 
BR-RN | Rio Grande do Norte | 
BR-RO | Rondônia | 
BR-RR | Roraima | 
BR-RS | Rio Grande do Sul | 
BR-SC | Santa Catarina | 
BR-SE | Sergipe | 
BR-SP | São Paulo | 
BR-TO | Tocatins | 
BR-U-A | (unassigned) | 
(28 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- `COUNT(*)` et `WHERE` se combinent : le SGBD **filtre** d'abord (les régions du Brésil), **puis compte**.
- Dans la relation `Regions`, l'attribut `country` contient le **code ISO** du pays (`'BR'`), pas son nom : `WHERE country = 'Brazil'` donnerait 0.

## 3 (e)

#### 📝 ÉNONCÉ

> (e) Combien d'aéroports ont une altitude négative ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Airports WHERE altitude < 0;
```

**Réponse :** 90 aéroports.

*Exécution dans le notebook :*

In [25]:
affiche("""
SELECT COUNT(*) FROM Airports WHERE altitude < 0;
""")

COUNT(*) | 
90 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [26]:
assert resultat("SELECT COUNT(*) FROM Airports WHERE altitude < 0") == [(90,)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- Les comparaisons `<`, `>`, `<=`, `>=` s'utilisent comme en Python ; « différent de » s'écrit `<>` en SQL.
- Une altitude négative signifie que l'aéroport est **sous le niveau de la mer** (par exemple Amsterdam-Schiphol, ou autour de la mer Morte).

## 3 (f)

#### 📝 ÉNONCÉ

> (f) Combien la France compte-t-elle d'héliports ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Airports WHERE country = 'FR' AND type = 'heliport';
```

**Réponse :** 265 héliports.

*Exécution dans le notebook :*

In [27]:
affiche("""
SELECT COUNT(*) FROM Airports WHERE country = 'FR' AND type = 'heliport';
""")

COUNT(*) | 
265 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [28]:
assert resultat("SELECT COUNT(*) FROM Airports WHERE country = 'FR' AND type = 'heliport'") == [(265,)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- Même schéma qu'en (c) et (d) : un filtre avec `AND`, puis un comptage.

## 3 (g)

#### 📝 ÉNONCÉ

> (g) Quelles sont les longueurs des pistes de l'aéroport Charles de Gaulle ?

#### ✅ RÉSOLUTION

Les pistes sont dans la relation `Runways`, mais l'aéroport y est désigné par son **identifiant** `airport_id`. On cherche d'abord cet identifiant (code ICAO trouvé avec `name LIKE '%Gaulle%'`), puis les pistes :

```sql
SELECT id FROM Airports WHERE code = 'LFPG';
SELECT length FROM Runways WHERE airport_id = 4185;
```

**Réponse :** 4 pistes : 13 829, 8 858, 8 858 et 13 780 pieds (soit environ 4 215 m, 2 700 m, 2 700 m et 4 200 m).

*Vérification (non demandée) :*

In [29]:
affiche("SELECT id, code, name FROM Airports WHERE name LIKE '%Gaulle%'")
affiche("SELECT length FROM Runways WHERE airport_id = 4185")
assert resultat("SELECT id FROM Airports WHERE code = 'LFPG'") == [(4185,)]
assert sorted(resultat("SELECT length FROM Runways WHERE airport_id = 4185")) == [(8858,), (8858,), (13780,), (13829,)]

id | code | name | 
4185 | LFPG | Charles de Gaulle International Airport | 
(1 ligne(s))
length | 
13829 | 
8858 | 
8858 | 
13780 | 
(4 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- Les informations sont réparties dans **deux relations** : l'aéroport dans `Airports`, ses pistes dans `Runways`. Le lien entre les deux est l'attribut `airport_id` de `Runways`, qui contient l'`id` d'un aéroport : c'est une **clé étrangère**.
- Ici, on fait le lien « à la main » en deux requêtes. La question 4 montre comment le faire en une seule requête avec `JOIN`.
- Deux pistes ont la même longueur (8 858) : on les voit toutes les deux, car `SELECT` sans `DISTINCT` garde les doublons, ce qui est voulu ici (ce sont deux pistes différentes).

## 3 (h)

#### 📝 ÉNONCÉ

> (h) Dans quel pays se trouve l'aéroport le plus au sud ?

#### ✅ RÉSOLUTION

Le plus au sud, c'est la **plus petite latitude**. En trois étapes :

```sql
SELECT MIN(latitude) FROM Airports;
SELECT name, country FROM Airports WHERE latitude = -90;
SELECT name FROM Countries WHERE code = 'AQ';
```

**Réponse :** *Antarctica* (l'Antarctique) : c'est l'aéroport de la station Amundsen-Scott, exactement au pôle Sud (latitude −90).

*Vérification (non demandée) :*

In [30]:
affiche("SELECT MIN(latitude) FROM Airports")
affiche("SELECT name, country FROM Airports WHERE latitude = -90")
affiche("SELECT name FROM Countries WHERE code = 'AQ'")
assert resultat("SELECT name FROM Countries WHERE code = 'AQ'") == [("Antarctica",)]

MIN(latitude) | 
-90.0 | 
(1 ligne(s))
name | country | 


South Pole Station Airport | AQ | 
(1 ligne(s))
name | 
Antarctica | 
(1 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- La latitude va de −90 (pôle Sud) à +90 (pôle Nord) : « le plus au sud » = `MIN(latitude)`.
- ⚠️ On ne peut **pas** écrire `SELECT country, MIN(latitude) FROM Airports;` en espérant obtenir le pays de l'aéroport le plus au sud : mélanger un attribut et une fonction d'agrégation sans regroupement est une erreur dans H2 (et donne un résultat non garanti dans d'autres SGBD).
- Le pays obtenu dans `Airports` est un **code** (`AQ`) : il faut une troisième requête (ou une jointure) pour obtenir son nom.

## 3 (i)

#### 📝 ÉNONCÉ

> (i) Quelle est la ville où se trouve la piste en asphalte la plus longue ?

#### ✅ RÉSOLUTION

```sql
SELECT MAX(length) FROM Runways WHERE type = 'ASP';
SELECT airport_id FROM Runways WHERE type = 'ASP' AND length = 16076;
SELECT city FROM Airports WHERE id = 2847;
```

**Réponse :** *Upington* (Afrique du Sud) : piste de 16 076 pieds (environ 4,9 km).

*Vérification (non demandée) :*

In [31]:
affiche("SELECT MAX(length) FROM Runways WHERE type = 'ASP'")
affiche("SELECT airport_id FROM Runways WHERE type = 'ASP' AND length = 16076")
affiche("SELECT city FROM Airports WHERE id = 2847")
assert resultat("SELECT city FROM Airports WHERE id = 2847") == [("Upington",)]

MAX(length) | 
16076 | 
(1 ligne(s))
airport_id | 


2847 | 
(1 ligne(s))
city | 


Upington | 
(1 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- Le sujet indique « asp : asphalte », mais dans les données les codes sont **en majuscules** : `'ASP'`. Avec `type = 'asp'`, on ne trouverait rien. On le découvre avec `SELECT DISTINCT type FROM Runways;`.
- **Regard critique :** les données d'*OurAirports* sont saisies par des contributeurs, et on trouve aussi `'ASPH'`, `'Asphalt'`, `'asphalt'`... pour la même chose. On s'en tient ici au code donné par le sujet.
- Même méthode qu'en (h) : on calcule d'abord le maximum, puis on cherche l'enregistrement qui a cette valeur.

## 3 (j)

#### 📝 ÉNONCÉ

> (j) Quelle est la ville d'Europe où se trouve la piste en asphalte la plus longue ?

#### ✅ RÉSOLUTION

Le continent n'est pas dans `Runways`, ni dans `Airports` : il est dans `Countries`. Il faut donc relier les **trois** relations avec des jointures :
- `Runways.airport_id` fait référence à `Airports.id` ;
- `Airports.country` fait référence à `Countries.code`.

```sql
SELECT MAX(Runways.length) FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP' AND Countries.continent = 'EU';
SELECT Airports.city FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP' AND Countries.continent = 'EU' AND Runways.length = 13829;
```

**Réponse :** *Paris* (aéroport Charles de Gaulle, piste de 13 829 pieds, soit 4 215 m).

*Vérification (non demandée) :*

In [32]:
affiche("""
SELECT MAX(Runways.length) FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP' AND Countries.continent = 'EU';
""")
affiche("""
SELECT Airports.city FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP' AND Countries.continent = 'EU' AND Runways.length = 13829;
""")

MAX(Runways.length) | 
13829 | 
(1 ligne(s))


city | 
Paris | 
(1 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- **La jointure** `R1 JOIN R2 ON condition` met « côte à côte » les enregistrements de R1 et de R2 qui vérifient la condition. Ici, chaque piste est collée à son aéroport, puis au pays de cet aéroport : on peut alors filtrer sur des attributs des trois relations à la fois.
- Quand plusieurs relations ont un attribut du même nom (`id`, `code`, `name`, `type`, `country`...), on **préfixe** par le nom de la relation : `Runways.type` (type de piste) n'est pas `Airports.type` (type d'aéroport).
- ⚠️ La condition `Countries.continent = 'EU'` doit figurer dans les **deux** requêtes : sinon, une piste de 13 829 pieds hors d'Europe serait aussi trouvée.

---
# QUESTION 4

#### 📝 ÉNONCÉ

> 4. Si ce n'était pas le cas, répondre aux questions précédentes en utilisant une seule requête SQL.

Les questions (a), (b), (e) et (f) sont déjà résolues en une seule requête. Il reste (c), (d), (g), (h), (i) et (j). Deux outils :
- la **jointure** (`JOIN`), au programme, pour aller chercher une information dans une autre relation au lieu de recopier un code ou un identifiant à la main ;
- la **requête imbriquée** (une requête entre parenthèses à l'intérieur d'une autre), **hors programme** mais présentée page 8 du polycopié, pour utiliser un maximum ou un minimum sans le recopier.

## 4 — (c) héliports de Berlin

#### 📝 ÉNONCÉ

> (c) Quels sont les noms des héliports de la capitale de l'Allemagne ?

#### ✅ RÉSOLUTION

```sql
SELECT Airports.name FROM Airports
    JOIN Countries ON Countries.code = Airports.country
    WHERE Airports.type = 'heliport' AND Airports.city = 'Berlin' AND Countries.name = 'Germany';
```

*Exécution dans le notebook :*

In [33]:
affiche("""
SELECT Airports.name FROM Airports
    JOIN Countries ON Countries.code = Airports.country
    WHERE Airports.type = 'heliport' AND Airports.city = 'Berlin' AND Countries.name = 'Germany';
""")

name | 
Berlin Emergency Hospital Heliport | 
Helios Clinic Berlin-Buch Heliport | 
Vivantes Humboldt Clinic Heliport | 
Charité Clinic Virchow Campus Heliport | 
Vivantes Freidrichshain Clinic Heliport | 
Berlin Tempelhof Heliport | 
Charité Clinic Benjamin Franklin Campus Heliport | 
Havelhöhe Medical Heliport | 
(8 ligne(s))


*Vérification (non demandée) :*

In [34]:
assert len(resultat("""
SELECT Airports.name FROM Airports
    JOIN Countries ON Countries.code = Airports.country
    WHERE Airports.type = 'heliport' AND Airports.city = 'Berlin' AND Countries.name = 'Germany';
""")) == 8

#### 💡 EXPLICATION PÉDAGOGIQUE

- Au lieu de chercher le code `DE` puis de le recopier, on **joint** `Airports` et `Countries` et on filtre directement sur le nom du pays.
- `Airports.name` (nom de l'aéroport) et `Countries.name` (nom du pays) : le préfixe est **obligatoire**, sinon le SGBD ne sait pas de quel `name` on parle (erreur « ambiguous column »).

## 4 — (d) régions du Brésil

#### 📝 ÉNONCÉ

> (d) Combien de régions sont répertoriées au Brésil ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Regions
    JOIN Countries ON Countries.code = Regions.country
    WHERE Countries.name = 'Brazil';
```

*Exécution dans le notebook :*

In [35]:
affiche("""
SELECT COUNT(*) FROM Regions
    JOIN Countries ON Countries.code = Regions.country
    WHERE Countries.name = 'Brazil';
""")

COUNT(*) | 
28 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [36]:
assert resultat("""
SELECT COUNT(*) FROM Regions
    JOIN Countries ON Countries.code = Regions.country
    WHERE Countries.name = 'Brazil';
""") == [(28,)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- Même idée : `Regions.country` contient un code de pays qui fait référence à `Countries.code`.

## 4 — (g) pistes de Charles de Gaulle

#### 📝 ÉNONCÉ

> (g) Quelles sont les longueurs des pistes de l'aéroport Charles de Gaulle ?

#### ✅ RÉSOLUTION

```sql
SELECT Runways.length FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    WHERE Airports.code = 'LFPG';
```

*Exécution dans le notebook :*

In [37]:
affiche("""
SELECT Runways.length FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    WHERE Airports.code = 'LFPG';
""")

length | 
8858 | 
8858 | 
13780 | 
13829 | 
(4 ligne(s))


*Vérification (non demandée) :*

In [38]:
assert sorted(resultat("""
SELECT Runways.length FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    WHERE Airports.code = 'LFPG';
""")) == [(8858,), (8858,), (13780,), (13829,)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- La condition de jointure `Airports.id = Runways.airport_id` relie chaque piste à **son** aéroport : c'est la clé étrangère `airport_id` qui fait référence à la clé `id`.
- ⚠️ **Erreur fréquente :** oublier le `ON ...`, ou écrire une mauvaise condition (`Airports.id = Runways.id`) : le SGBD associerait alors des pistes à des aéroports qui n'ont rien à voir.

## 4 — (h) aéroport le plus au sud

#### 📝 ÉNONCÉ

> (h) Dans quel pays se trouve l'aéroport le plus au sud ?

#### ✅ RÉSOLUTION

```sql
SELECT Countries.name FROM Countries
    JOIN Airports ON Countries.code = Airports.country
    WHERE Airports.latitude = (SELECT MIN(latitude) FROM Airports);
```

*Exécution dans le notebook :*

In [39]:
affiche("""
SELECT Countries.name FROM Countries
    JOIN Airports ON Countries.code = Airports.country
    WHERE Airports.latitude = (SELECT MIN(latitude) FROM Airports);
""")

name | 
Antarctica | 
(1 ligne(s))


*Vérification (non demandée) :*

In [40]:
assert resultat("""
SELECT Countries.name FROM Countries
    JOIN Airports ON Countries.code = Airports.country
    WHERE Airports.latitude = (SELECT MIN(latitude) FROM Airports);
""") == [("Antarctica",)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Requête imbriquée :** le SGBD calcule d'abord `(SELECT MIN(latitude) FROM Airports)`, qui vaut −90, puis l'utilise comme une valeur ordinaire dans le `WHERE`. On n'a plus besoin de recopier −90 à la main.
- La jointure avec `Countries` remplace la troisième requête (code `AQ` → nom du pays).

## 4 — (i) piste en asphalte la plus longue

#### 📝 ÉNONCÉ

> (i) Quelle est la ville où se trouve la piste en asphalte la plus longue ?

#### ✅ RÉSOLUTION

```sql
SELECT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Runways.type = 'ASP'
      AND Runways.length = (SELECT MAX(length) FROM Runways WHERE type = 'ASP');
```

*Exécution dans le notebook :*

In [41]:
affiche("""
SELECT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Runways.type = 'ASP'
      AND Runways.length = (SELECT MAX(length) FROM Runways WHERE type = 'ASP');
""")

city | 
Upington | 
(1 ligne(s))


*Vérification (non demandée) :*

In [42]:
assert resultat("""
SELECT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Runways.type = 'ASP'
      AND Runways.length = (SELECT MAX(length) FROM Runways WHERE type = 'ASP');
""") == [("Upington",)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- ⚠️ La condition `Runways.type = 'ASP'` doit apparaître **dans la requête imbriquée** (pour calculer le maximum des pistes en asphalte, et non de toutes les pistes, qui vaut 30 000) **et dans la requête principale** (pour ne garder que la piste en asphalte qui a cette longueur).

## 4 — (j) en Europe

#### 📝 ÉNONCÉ

> (j) Quelle est la ville d'Europe où se trouve la piste en asphalte la plus longue ?

#### ✅ RÉSOLUTION

```sql
SELECT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP' AND Countries.continent = 'EU'
      AND Runways.length = (SELECT MAX(Runways.length) FROM Runways
                                JOIN Airports ON Airports.id = Runways.airport_id
                                JOIN Countries ON Countries.code = Airports.country
                                WHERE Runways.type = 'ASP' AND Countries.continent = 'EU');
```

*Exécution dans le notebook :*

In [43]:
affiche("""
SELECT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP' AND Countries.continent = 'EU'
      AND Runways.length = (SELECT MAX(Runways.length) FROM Runways
                                JOIN Airports ON Airports.id = Runways.airport_id
                                JOIN Countries ON Countries.code = Airports.country
                                WHERE Runways.type = 'ASP' AND Countries.continent = 'EU');
""")

city | 
Paris | 
(1 ligne(s))


*Vérification (non demandée) :*

In [44]:
assert resultat("""
SELECT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP' AND Countries.continent = 'EU'
      AND Runways.length = (SELECT MAX(Runways.length) FROM Runways
                                JOIN Airports ON Airports.id = Runways.airport_id
                                JOIN Countries ON Countries.code = Airports.country
                                WHERE Runways.type = 'ASP' AND Countries.continent = 'EU');
""") == [("Paris",)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- C'est la combinaison des deux outils : les deux requêtes de la question 3 (j) emboîtées l'une dans l'autre.
- La requête imbriquée a besoin **des mêmes jointures** que la requête principale, puisque le continent est dans `Countries`.
- Conseil : écrire et tester d'abord la requête intérieure seule, puis l'emboîter.

---
# QUESTION 5

#### 📝 ÉNONCÉ

> 5. Télécharger et exécuter le fichier airports_tables_v2.sql et importer de nouveau les données du fichier airports_data.sql.

#### ✅ RÉSOLUTION

⏳ **Non traitée pour l'instant :** le lien vers `airports_tables_v2.sql` sur le site du professeur ne fonctionne pas encore (fichier absent du serveur). La méthode serait la même qu'à la question 1, avec `airports_tables_v2.sql` à la place de `airports_tables.sql`.

Les questions de la partie 6 n'utilisent que des attributs qui existent déjà dans les tables actuelles : elles sont résolues ci-dessous avec les données déjà chargées.

---
# QUESTION 6

> 6. Une autre série de questions...

## 6 (a)

#### 📝 ÉNONCÉ

> (a) Quel est le nombre de pistes de largeur supérieure ou égale à 100 m situées aux Seychelles ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Countries.name = 'Seychelles' AND Runways.width >= 100;
```

**Réponse :** 3 pistes (en prenant la valeur 100 comme le sujet).

*Exécution dans le notebook :*

In [45]:
affiche("""
SELECT COUNT(*) FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Countries.name = 'Seychelles' AND Runways.width >= 100;
""")

COUNT(*) | 
3 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [46]:
assert resultat("""
SELECT COUNT(*) FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Countries.name = 'Seychelles' AND Runways.width >= 100;
""") == [(3,)]
affiche("""
SELECT Airports.name, Runways.width FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    WHERE Airports.country = 'SC';
""")

name | width | 
Alphonse Airport | 75 | 
Assumption Island Airport | 100 | 
Desroches Airport | 33 | 
Farquhar Airport | 75 | 
Seychelles International Airport | 151 | 
Praslin Airport | 104 | 
Coetivy Airport | 0 | 
(7 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- Deux jointures pour relier une piste à son pays : `Runways` → `Airports` → `Countries`.
- `>=` signifie « supérieur ou égal » : la piste de largeur exactement 100 est comptée.
- **Remarque sur l'unité :** dans *OurAirports*, les largeurs sont **en pieds**. 100 m ≈ 328 pieds, et aucune piste des Seychelles n'est aussi large (la plus large fait 151 pieds ≈ 46 m). Avec `Runways.width >= 328`, la réponse serait 0. On suit ici l'énoncé, qui prend 100 comme valeur.

## 6 (b)

#### 📝 ÉNONCÉ

> (b) Quelles sont les villes françaises, classées par ordre alphabétique, qui ont un aéroport avec (au moins) une piste en herbe ?

#### ✅ RÉSOLUTION

```sql
SELECT DISTINCT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Airports.country = 'FR' AND Runways.type = 'GRS'
    ORDER BY Airports.city;
```

**Réponse :** 38 lignes, d'*Ambérieu* à *Île d'Yeu* (la première ligne est vide : un aéroport dont la ville n'est pas renseignée).

*Exécution dans le notebook :*

In [47]:
affiche("""
SELECT DISTINCT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Airports.country = 'FR' AND Runways.type = 'GRS'
    ORDER BY Airports.city;
""")

city | 
 | 
Ambérieu | 
Arcachon/La Teste-de-Buch | 
Autun/Bellevue | 
Avaux | 
Avernes | 
Belpech | 
Corbeny | 
Coulommiers/Voisins | 
Fontrailles | 
Haguenau | 
Kogenheim | 
La Tranche sur Mer | 
Lanvéoc/Poulmic | 
Larreule | 
Le Bez | 
Le Fousseret | 
Le Gault-Saint-Denis | 
Lurcy-Lévis | 
Maisse | 
... (seules les 20 premières lignes sont affichées)
(38 ligne(s))


*Vérification (non demandée) :*

In [48]:
assert len(resultat("""
SELECT DISTINCT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Airports.country = 'FR' AND Runways.type = 'GRS'
    ORDER BY Airports.city;
""")) == 38

#### 💡 EXPLICATION PÉDAGOGIQUE

- `ORDER BY attribut` **trie** le résultat (ordre croissant, donc alphabétique pour du texte). Il se place **à la fin** de la requête, après le `WHERE`.
- `DISTINCT` est indispensable : un aéroport peut avoir **plusieurs** pistes en herbe, et une ville plusieurs aéroports ; sans `DISTINCT`, la même ville apparaîtrait plusieurs fois.
- Herbe = `'GRS'` (code donné par le sujet, en majuscules dans les données).
- Les noms commençant par une lettre accentuée (*Écrosnes*, *Évaux*, *Île d'Yeu*) sont rangés **après le Z** : le tri compare les codes des caractères, et `É` a un code plus grand que `Z`.

## 6 (c)

#### 📝 ÉNONCÉ

> (c) Quels sont les noms des pays qui disposent de petits aéroports avec des pistes en gravier ?

#### ✅ RÉSOLUTION

Le sujet ne donne pas le code du gravier : on le cherche dans les données (`SELECT DISTINCT type FROM Runways;`). Le code le plus utilisé est `'GVL'` (*gravel*).

```sql
SELECT DISTINCT Countries.name FROM Countries
    JOIN Airports ON Countries.code = Airports.country
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Airports.type = 'small_airport' AND Runways.type = 'GVL';
```

**Réponse :** 28 pays (Australia, Belize, Brazil, ..., United States).

*Exécution dans le notebook :*

In [49]:
affiche("""
SELECT DISTINCT Countries.name FROM Countries
    JOIN Airports ON Countries.code = Airports.country
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Airports.type = 'small_airport' AND Runways.type = 'GVL';
""")

name | 
United States | 
Solomon Islands | 
Australia | 
Papua New Guinea | 
Greenland | 
Brazil | 
Belize | 
Canada | 
Democratic Republic of the Congo | 
Burkina Faso | 
Finland | 
Spain | 
South Africa | 
Eswatini | 
Mali | 
Guinea | 
Libya | 
Haiti | 
Tanzania | 
Uganda | 
... (seules les 20 premières lignes sont affichées)
(28 ligne(s))


*Vérification (non demandée) :*

In [50]:
assert len(resultat("""
SELECT DISTINCT Countries.name FROM Countries
    JOIN Airports ON Countries.code = Airports.country
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Airports.type = 'small_airport' AND Runways.type = 'GVL';
""")) == 28

#### 💡 EXPLICATION PÉDAGOGIQUE

- « Petits aéroports » = `Airports.type = 'small_airport'` ; « pistes en gravier » = `Runways.type = 'GVL'`. Le préfixe est indispensable : les deux relations ont un attribut `type`.
- **Regard critique :** le gravier est aussi codé `'GRVL'`, `'GRAVEL'`, `'Gravel'`... selon les contributeurs. Pour les inclure, on pourrait écrire `Runways.type IN ('GVL', 'GRVL', 'GRAVEL')` (le mot-clé `IN` teste l'appartenance à une liste ; il est hors programme).

## 6 (d)

#### 📝 ÉNONCÉ

> (d) Quelles sont les villes européennes qui disposent de pistes en asphalte éclairées ?

#### ✅ RÉSOLUTION

```sql
SELECT DISTINCT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Countries.continent = 'EU' AND Runways.type = 'ASP' AND Runways.lighted = 1;
```

**Réponse :** 762 villes (Aalborg, Aberdeen, ...). Le nombre s'obtient en remplaçant `DISTINCT Airports.city` par `COUNT(DISTINCT Airports.city)`.

*Exécution dans le notebook :*

In [51]:
affiche("""
SELECT DISTINCT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Countries.continent = 'EU' AND Runways.type = 'ASP' AND Runways.lighted = 1;
""")

city | 
Gmünd | 
Akureyri | 
Egilsstaðir | 
Reykjavík | 
Reykjavik | 
Prishtina | 
Berlin | 
Antwerp | 
Zaventem | 
Brussels | 
Chièvres | 
Florennes | 
Wevelgem | 
Liège | 
Dessau | 
Eisenhüttenstadt | 
Heringsdorf | 
Oppin | 
Trebbin | 
 | 
... (seules les 20 premières lignes sont affichées)
(762 ligne(s))


*Vérification (non demandée) :*

In [52]:
assert len(resultat("""
SELECT DISTINCT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Countries.continent = 'EU' AND Runways.type = 'ASP' AND Runways.lighted = 1;
""")) == 762

#### 💡 EXPLICATION PÉDAGOGIQUE

- Trois conditions reliées par `AND`, sur des attributs de trois relations différentes : c'est possible grâce aux deux jointures.
- « Éclairée » : d'après le sujet, `lighted` vaut **1** si la piste est éclairée (0 si non, −1 si inconnu). On écrit donc `lighted = 1`, et non `lighted = 'oui'` ou `lighted = True`.

## 6 (e)

#### 📝 ÉNONCÉ

> (e) Dans quelle ville de l'hémisphère sud se trouve l'aéroport comportant la plus longue piste ?

#### ✅ RÉSOLUTION

```sql
SELECT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Airports.latitude < 0
      AND Runways.length = (SELECT MAX(Runways.length) FROM Runways
                                JOIN Airports ON Airports.id = Runways.airport_id
                                WHERE Airports.latitude < 0);
```

**Réponse :** *Upington* (Afrique du Sud).

*Exécution dans le notebook :*

In [53]:
affiche("""
SELECT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Airports.latitude < 0
      AND Runways.length = (SELECT MAX(Runways.length) FROM Runways
                                JOIN Airports ON Airports.id = Runways.airport_id
                                WHERE Airports.latitude < 0);
""")

city | 
Upington | 
(1 ligne(s))


*Vérification (non demandée) :*

In [54]:
assert resultat("""
SELECT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Airports.latitude < 0
      AND Runways.length = (SELECT MAX(Runways.length) FROM Runways
                                JOIN Airports ON Airports.id = Runways.airport_id
                                WHERE Airports.latitude < 0);
""") == [("Upington",)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Hémisphère sud** = sous l'équateur = **latitude négative** : `Airports.latitude < 0`.
- Même structure que la question 4 (i) : une requête imbriquée calcule la longueur maximale parmi les pistes de l'hémisphère sud, puis on cherche la ville correspondante.
- On retrouve Upington : sa piste en asphalte est aussi la plus longue de l'hémisphère sud, tous types confondus.

## 6 (f)

#### 📝 ÉNONCÉ

> (f) Quels sont, pour chaque continent, les nombres cumulés de kilomètres de pistes en asphalte ?

#### ✅ RÉSOLUTION

```sql
SELECT Countries.continent, SUM(Runways.length) * 0.3048 / 1000 FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP'
    GROUP BY Countries.continent;
```

**Réponse :** environ 1 751 km (AF), 2 768 km (AS), 2 460 km (EU), 7 598 km (NA), 789 km (OC), 2 800 km (SA). Aucune piste en asphalte n'est répertoriée en Antarctique avec ce code.

*Exécution dans le notebook :*

In [55]:
affiche("""
SELECT Countries.continent, SUM(Runways.length) * 0.3048 / 1000 FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP'
    GROUP BY Countries.continent;
""")

continent | SUM(Runways.length) * 0.3048 / 1000 | 
AF | 1751.0528352000001 | 
AS | 2767.8647208 | 
EU | 2460.0188544000002 | 
NA | 7598.4686232 | 
OC | 789.1781016000001 | 
SA | 2800.2576455999997 | 
(6 ligne(s))


*Vérification (non demandée) :*

In [56]:
assert len(resultat("""
SELECT Countries.continent, SUM(Runways.length) * 0.3048 / 1000 FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP'
    GROUP BY Countries.continent;
""")) == 6

#### 💡 EXPLICATION PÉDAGOGIQUE

- `SUM(...)` additionne les longueurs. Mais on veut **une somme par continent**, pas une seule somme pour le monde entier.
- **`GROUP BY` (hors programme)** regroupe les enregistrements qui ont la même valeur de `continent`, puis applique `SUM` **à chaque groupe** : on obtient une ligne par continent.
- **Conversion :** les longueurs sont en pieds ; × 0,3048 donne des mètres, puis ÷ 1000 des kilomètres. On peut faire des calculs directement dans le `SELECT`.
- Sans le `GROUP BY`, H2 refuse la requête : on ne peut pas afficher `continent` (une valeur par ligne) à côté de `SUM` (une seule valeur) sans dire comment regrouper.

## 6 (g)

#### 📝 ÉNONCÉ

> (g) Quelle est la région de France dénombrant le plus d'aéroports ?

#### ✅ RÉSOLUTION

```sql
SELECT Regions.name, COUNT(*) FROM Regions
    JOIN Airports ON Regions.code = Airports.region
    WHERE Regions.country = 'FR'
    GROUP BY Regions.name
    ORDER BY COUNT(*) DESC
    LIMIT 1;
```

**Réponse :** *Auvergne-Rhône-Alpes*, avec 151 aéroports (tous types confondus, héliports et aéroports fermés compris).

*Exécution dans le notebook :*

In [57]:
affiche("""
SELECT Regions.name, COUNT(*) FROM Regions
    JOIN Airports ON Regions.code = Airports.region
    WHERE Regions.country = 'FR'
    GROUP BY Regions.name
    ORDER BY COUNT(*) DESC
    LIMIT 1;
""")

name | COUNT(*) | 
Auvergne-Rhône-Alpes | 151 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [58]:
assert resultat("""
SELECT Regions.name, COUNT(*) FROM Regions
    JOIN Airports ON Regions.code = Airports.region
    WHERE Regions.country = 'FR'
    GROUP BY Regions.name
    ORDER BY COUNT(*) DESC
    LIMIT 1;
""") == [("Auvergne-Rhône-Alpes", 151)]

# Le classement complet (sans LIMIT) :
affiche("""
SELECT Regions.name, COUNT(*) FROM Regions
    JOIN Airports ON Regions.code = Airports.region
    WHERE Regions.country = 'FR'
    GROUP BY Regions.name
    ORDER BY COUNT(*) DESC;
""")

name | COUNT(*) | 
Auvergne-Rhône-Alpes | 151 | 
Occitanie | 139 | 
Nouvelle-Aquitaine | 117 | 
Grand Est | 102 | 
Provence-Alpes-Côte-d'Azur | 84 | 
Bourgogne-Franche-Comté | 68 | 
Hauts-de-France | 67 | 
Normandy | 56 | 
Île-de-France | 50 | 
Centre-Val de Loire | 45 | 
Pays-de-la-Loire | 43 | 
Brittany | 40 | 
Corsica | 11 | 
(unassigned) | 10 | 
(14 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- `GROUP BY Regions.name` + `COUNT(*)` : on compte les aéroports **de chaque région**.
- `ORDER BY COUNT(*) DESC` trie du plus grand au plus petit nombre (`DESC` = décroissant ; par défaut, l'ordre est croissant).
- `LIMIT 1` (hors programme) ne garde que la première ligne. Sans `LIMIT`, on lit simplement la première ligne du classement.
- La jointure se fait sur `Regions.code = Airports.region` : dans `Airports`, `region` contient le code de la région (`'FR-ARA'`), et non son nom.